In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [2]:
data = pd.read_csv(r"C:\ML project\Reviews.csv")

In [3]:
data = data[['Summary', 'Text', 'Score']]
data['Summary'].fillna('', inplace=True)
data['Text'].fillna('', inplace=True)

In [4]:
data['review'] = data['Summary'] + " " + data['Text']

In [5]:
def convert_sentiment(x):
    if x >= 4:
        return "positive"
    elif x == 3:
        return "neutral"
    else:
        return "negative"

data['sentiment'] = data['Score'].apply(convert_sentiment)


data = data[['review', 'sentiment']]


In [6]:
positive = data[data['sentiment'] == 'positive']
negative = data[data['sentiment'] == 'negative']
neutral = data[data['sentiment'] == 'neutral']
min_count = min(len(positive), len(negative), len(neutral))
positive = positive.sample(min_count, random_state=42)
negative = negative.sample(min_count, random_state=42)
neutral = neutral.sample(min_count, random_state=42)
data = pd.concat([positive, negative, neutral])
data = data.sample(frac=1, random_state=42)

In [7]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=5000)
X = tfidf.fit_transform(data['review'])
y = data['sentiment']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

C:\Users\Sagudam Girish\anaconda3\envs\python\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


LogisticRegression(max_iter=200)

In [10]:
print("Accuracy:", model.score(X_test, y_test)*100)

Accuracy: 75.05081300813008


In [11]:
def predict_review(text):
    vec = tfidf.transform([text])
    return model.predict(vec)[0]

In [12]:
print(predict_review("this product is amazing"))
print(predict_review("this product is bad"))
print(predict_review("very poor quality"))

positive
negative
negative


In [13]:
while True:
    user_input = input("Enter review (or 'exit'): ")
    
    if user_input == "exit":
        break
    
    result = predict_review(user_input)
    print("Sentiment:", result)

Enter review (or 'exit'): its good 
Sentiment: positive
Enter review (or 'exit'): i liked it
Sentiment: neutral
Enter review (or 'exit'): i am good
Sentiment: positive
Enter review (or 'exit'): its great
Sentiment: positive
Enter review (or 'exit'): exit


In [14]:
new_reviews = []
while True:
    user_input = input("Enter review (or 'exit'): ")
    if user_input == "exit":
        break
    result = predict_review(user_input)
    new_reviews.append((user_input, result))
    print("Sentiment:", result)

Enter review (or 'exit'): great
Sentiment: positive
Enter review (or 'exit'): i liked it
Sentiment: neutral
Enter review (or 'exit'): its good
Sentiment: positive
Enter review (or 'exit'): exit


In [15]:
new_df = pd.DataFrame(new_reviews, columns=['review', 'sentiment'])

In [16]:
print(new_df['sentiment'].value_counts())

positive    2
neutral     1
Name: sentiment, dtype: int64


In [17]:
percent = new_df['sentiment'].value_counts(normalize=True) * 100
print(percent)

positive    66.666667
neutral     33.333333
Name: sentiment, dtype: float64


In [18]:
features = ["battery", "camera", "display", "performance", "price"]

In [19]:
feature_results = []

for review, sentiment in new_reviews:
    review_lower = review.lower()
    
    for feature in features:
        if feature in review_lower:
            feature_results.append((feature, sentiment))

In [20]:
feature_df = pd.DataFrame(feature_results, columns=['feature', 'sentiment'])

In [21]:
print(feature_df.groupby('feature')['sentiment'].value_counts())

Series([], Name: sentiment, dtype: int64)


In [22]:
feature_percent = feature_df.groupby('feature')['sentiment'].value_counts(normalize=True) * 100
print(feature_percent)

Series([], Name: sentiment, dtype: float64)


In [23]:
new_df = pd.DataFrame(new_reviews, columns=['review', 'sentiment'])

In [24]:
overall = new_df['sentiment'].value_counts(normalize=True) * 100

In [25]:
print(overall)

positive    66.666667
neutral     33.333333
Name: sentiment, dtype: float64


In [26]:
features = ["battery", "camera", "display", "performance", "price"]

feature_results = []

for review, sentiment in zip(data['review'], data['sentiment']):
    review_lower = review.lower()
    
    for feature in features:
        if feature in review_lower:
            feature_results.append((feature, sentiment))

feature_df = pd.DataFrame(feature_results, columns=['feature', 'sentiment'])

In [27]:
print(feature_df.head())

  feature sentiment
0   price  negative
1   price   neutral
2   price   neutral
3   price  negative
4   price  negative


In [28]:
from groq import Groq

client = Groq(api_key="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")

prompt = f"""
You are an AI assistant.

Analyze the following customer review data and give a clear summary.

Overall Sentiment:
{overall}

Feature Insights:
{feature_df.groupby('feature')['sentiment'].value_counts()}

Give a short and clear summary.
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**Summary of Customer Review Data:**

Based on the analysis, the overall sentiment is **positive** with 66.67% of customers expressing satisfaction with the product. Key features were evaluated as follows:

- Display had the most negative comments (112), but was also praised by 45 customers.
- Price was the most positively received feature (6379 comments), indicating that customers felt it was reasonable or good value for money.
- Battery and camera both received some negative feedback, but camera was also praised by 6 customers.
- Performance and display had a significant number of neutral comments, indicating some mixed opinions.

Overall, customers have some mixed feedback, but the positive sentiment and high number of price-related positive comments indicate that the product has its strengths.


In [31]:
import pickle
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))

In [32]:
print(type(tfidf))
print(type(model))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'sklearn.linear_model._logistic.LogisticRegression'>
